# 📖 Notebook 1: Message Delivery & Storage

Welcome to the first notebook in our **WhatsApp System Design Lab**! 🎉

In this notebook, we'll explore how a messaging app like WhatsApp actually delivers messages
from one person to another — even when the recipient is offline.

---

## 🎯 Learning Objectives

By the end of this notebook, you will understand:

1. **How messages are stored** in a relational database
2. **How WebSockets** enable real-time message delivery
3. **The Inbox Pattern** — how offline users receive messages later
4. **Redis Pub/Sub** — how real-time notifications work
5. **Sequence numbers** — how clients detect missed messages

## 🗺️ Where This Fits

```
📓 Notebook 1: Message Delivery & Storage   <-- YOU ARE HERE
📓 Notebook 2: Read Receipts & Presence
📓 Notebook 3: Group Messaging
📓 Notebook 4: End-to-End Encryption Basics
```

## 🏗️ Architecture Overview

```
┌──────────┐     WebSocket      ┌──────────────┐
│  Alice   │◄──────────────────►│              │
│  Phone   │                    │  Chat Server │
└──────────┘                    │  (Python)    │
                                │              │
┌──────────┐     WebSocket      │              │     ┌────────────┐
│  Bob     │◄──────────────────►│              │────►│ PostgreSQL │
│  Phone   │                    │              │     │ (Storage)  │
└──────────┘                    │              │     └────────────┘
                                │              │
┌──────────┐     WebSocket      │              │     ┌────────────┐
│  Diana   │◄──────────────────►│              │────►│   Redis    │
│  Phone   │                    │              │     │ (Pub/Sub)  │
└──────────┘                    └──────────────┘     └────────────┘
```

## ⚙️ Setup

Before running this notebook, make sure the infrastructure is running:

### 1. Start Docker services

```bash
cd system-designs/whatsapp
docker compose up -d
```

This starts PostgreSQL, Redis, the Chat Server, Adminer, and RedisInsight.

### 2. Select the right kernel

This notebook uses a **virtual environment** (`.venv`) with the required dependencies.

1. In VS Code, look at the **top-right** of this notebook for the kernel selector
2. Click it and choose the `.venv` kernel (from `system-designs/whatsapp/.venv`)
3. If you don't see it, run:
   ```bash
   cd system-designs/whatsapp
   uv venv
   source .venv/bin/activate
   uv sync
   ```
4. Then reload VS Code (`Cmd+Shift+P` → "Reload Window")

### 3. Visualization tools (optional but recommended)

- **Adminer** (PostgreSQL GUI): http://localhost:8080
  - System: PostgreSQL, Server: postgres, User: demo, Password: demo, Database: whatsapp_demo
- **RedisInsight** (Redis GUI): http://localhost:5540
  - Add connection: host=localhost, port=6379

In [1]:
# === 🔌 Connection Setup ===

import psycopg2
import psycopg2.extras
import redis
import json
import time
import threading
from websockets.sync.client import connect as ws_connect

# --- Configuration ---
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "whatsapp_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

WS_URL = "ws://localhost:8765"

# --- Helper Functions ---
def get_db():
    """Create a new database connection."""
    return psycopg2.connect(**DB_CONFIG)

def query(sql, params=None):
    """Run a SELECT query and return rows as dictionaries."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute(sql, params)
    rows = [dict(row) for row in cur.fetchall()]
    cur.close()
    conn.close()
    return rows

def execute(sql, params=None):
    """Run an INSERT/UPDATE/DELETE query."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute(sql, params)
    conn.commit()
    cur.close()
    conn.close()

def get_redis():
    """Create a new Redis connection."""
    return redis.Redis(**REDIS_CONFIG)

def print_table(rows, title=""):
    """Pretty-print a list of dicts as a table."""
    if not rows:
        print("  (no rows)")
        return
    if title:
        print(f"\n📋 {title}")
        print("─" * 60)
    keys = list(rows[0].keys())
    widths = {k: max(len(str(k)), max(len(str(r.get(k, ''))) for r in rows)) for k in keys}
    header = " | ".join(str(k).ljust(widths[k]) for k in keys)
    print(f"  {header}")
    print(f"  {'─' * len(header)}")
    for row in rows:
        line = " | ".join(str(row.get(k, '')).ljust(widths[k]) for k in keys)
        print(f"  {line}")

# --- Test Connections ---
print("🔌 Testing connections...\n")

try:
    rows = query("SELECT COUNT(*) as count FROM users")
    print(f"✅ PostgreSQL: Connected! ({rows[0]['count']} users found)")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    r = get_redis()
    r.ping()
    print(f"✅ Redis: Connected!")
except Exception as e:
    print(f"❌ Redis: {e}")

try:
    ws = ws_connect(WS_URL)
    ws.send(json.dumps({"type": "connect", "user_id": 1}))
    resp = json.loads(ws.recv())
    print(f"✅ WebSocket: Connected! (server says: {resp['type']})")
    ws.close()
except Exception as e:
    print(f"❌ WebSocket: {e}")

print("\n🎉 All connections ready! Let's explore message delivery.")

🔌 Testing connections...

✅ PostgreSQL: Connected! (5 users found)
✅ Redis: Connected!
✅ WebSocket: Connected! (server says: connected)

🎉 All connections ready! Let's explore message delivery.


---

# 🤔 The Problem: Why Is Message Delivery Hard?

Sending a text message sounds simple, right? You type "Hey!" and your friend sees it.

But think about all the things that can go wrong:

| Scenario | What happens? |
|----------|---------------|
| 📱 Bob is online | Message should appear **instantly** |
| 😴 Bob is offline | Message must be **saved** and delivered when Bob comes back |
| 📵 Bob's network drops mid-delivery | Message might be **lost** — we need to detect this |
| 💻📱 Bob has phone AND laptop | Message must reach **all** devices |
| 🔄 Messages arrive out of order | We need to **reorder** them correctly |

### 📮 Think of it like a post office

Imagine you're sending a letter to a friend:

```
📝 You write a letter (compose message)
    │
    ▼
📮 You drop it in the mailbox (send to server)
    │
    ▼
🏤 Post office receives it (server stores message)
    │
    ├──► 🏠 Friend is home? ──► Deliver immediately! ✅
    │
    └──► 🚫 Friend is away? ──► Hold in PO Box (inbox)
                                    │
                                    ▼
                               🏠 Friend comes home
                                    │
                                    ▼
                               📬 Picks up mail (sync)
                                    │
                                    ▼
                               ✅ Signs receipt (ACK)
```

This is exactly how our system works! Let's see how it's built.

---

# 🚫 Bad → ✅ Best: The Evolution of Message Delivery

Before we look at the _good_ design, let's understand **why** we chose it by walking through three designs — from naive to production-ready.

### ❌ v1 (BAD): HTTP Polling — "Did I get any new messages?"

```
Bob's phone every 5 seconds:
  GET /messages?since=123   -> Server: "nothing new"
  GET /messages?since=123   -> Server: "nothing new"
  GET /messages?since=123   -> Server: "1 new message!"
```

Problems:
- 📉 **Slow** — average 2.5s latency, up to 5s.
- 🔋 **Battery killer** — phone wakes up constantly.
- 💸 **Expensive** — 99% of requests return nothing.
- 📦 For 1B users polling every 5s → **200M requests/sec** just to ask "anything new?".

### ⚠️ v2 (BETTER): WebSocket only — no durable inbox

Replace polling with a live WebSocket. Now delivery is instant — but what if Bob is offline, or the notification drops mid-flight?

```
Alice -> Server -> (Bob offline) -> 💨 message vanishes
```

Redis pub/sub is **fire-and-forget**. No listener = no delivery. Messages get lost.

### ✅ v3 (BEST): WebSocket + Inbox table + Pub/Sub

Two layers working together:

| Layer | Technology | Role |
|-------|-----------|------|
| **Reliability** | `inbox` table in PostgreSQL | Remembers who still needs to receive each message |
| **Speed** | Redis pub/sub on `user:{id}` channels | Delivers instantly to online users |

If pub/sub drops it, the inbox still has it — the client resyncs when it reconnects. This is the design we're going to explore for the rest of the notebook. 👇


In [2]:
# 🔢 Back-of-envelope: polling vs WebSocket at scale

users = 1_000_000_000          # 1 billion users
poll_interval_sec = 5          # check every 5 seconds
msgs_per_user_per_day = 40     # realistic WhatsApp-like average

# --- v1: HTTP polling ---
polls_per_sec = users / poll_interval_sec
real_msgs_per_sec = users * msgs_per_user_per_day / 86400
wasted_pct = (1 - real_msgs_per_sec / polls_per_sec) * 100

print('v1 (BAD) - HTTP polling')
print(f'   Requests/sec:         {polls_per_sec:>20,.0f}')
print(f'   Wasted (no new msgs): {wasted_pct:>19.2f} %')
print(f'   Avg latency:          {poll_interval_sec/2:>20.1f} s')

# --- v3: WebSocket + inbox (only work on real events) ---
print('\nv3 (BEST) - WebSocket + inbox + pub/sub')
print(f'   Writes/sec (real):    {real_msgs_per_sec:>20,.0f}')
print(f'   Wasted work:          {0:>19} %')
print(f'   Avg latency:          {"~50ms":>20}')

print(f'\n💡 Polling does ~{polls_per_sec/real_msgs_per_sec:,.0f}x more work per delivered message.')


v1 (BAD) - HTTP polling
   Requests/sec:                  200,000,000
   Wasted (no new msgs):               99.77 %
   Avg latency:                           2.5 s

v3 (BEST) - WebSocket + inbox + pub/sub
   Writes/sec (real):                 462,963
   Wasted work:                            0 %
   Avg latency:                         ~50ms

💡 Polling does ~432x more work per delivered message.


---

# 🗄️ Exploring the Data Model

Before we send any messages, let's look at how data is organized in our database.

Our database has **6 tables**:

```
┌──────────────┐     ┌──────────────────┐     ┌──────────────┐
│    users     │     │ chat_participants │     │    chats     │
├──────────────┤     ├──────────────────┤     ├──────────────┤
│ id           │◄────│ user_id          │     │ id           │
│ username     │     │ chat_id          │───►│ name         │
│ display_name │     │ role             │     │ is_group     │
└──────────────┘     └──────────────────┘     └──────────────┘
                                                     │
┌──────────────┐     ┌──────────────────┐           │
│    inbox     │     │    messages       │           │
├──────────────┤     ├──────────────────┤           │
│ user_id      │     │ id               │           │
│ message_id   │───►│ chat_id          │◄──────────┘
│ status       │     │ sender_id        │
│ delivered_at │     │ content          │     ┌────────────────┐
└──────────────┘     │ sequence_number  │     │ chat_sequences │
                     └──────────────────┘     ├────────────────┤
                                              │ chat_id        │
                                              │ last_sequence  │
                                              └────────────────┘
```

Let's explore each one!

In [3]:
# 👥 Let's see who's in our system
users = query("SELECT id, username, display_name FROM users ORDER BY id")
print_table(users, "Users in our system")
print("\n💡 We have 5 test users. Think of them as 5 friends with the app installed.")


📋 Users in our system
────────────────────────────────────────────────────────────
  id | username | display_name 
  ─────────────────────────────
  1  | alice    | Alice Johnson
  2  | bob      | Bob Smith    
  3  | charlie  | Charlie Brown
  4  | diana    | Diana Prince 
  5  | eve      | Eve Wilson   

💡 We have 5 test users. Think of them as 5 friends with the app installed.


In [4]:
# 💬 Let's see the conversations (chats)
chats = query("""
    SELECT c.id, c.name, c.is_group,
           string_agg(u.username, ', ' ORDER BY u.id) as participants
    FROM chats c
    JOIN chat_participants cp ON c.id = cp.chat_id
    JOIN users u ON cp.user_id = u.id
    GROUP BY c.id, c.name, c.is_group
    ORDER BY c.id
""")
print_table(chats, "Chats (conversations)")

print("\n💡 Notice:")
print("   • Chats 1 & 2 are 1:1 chats (is_group = False, name is empty)")
print("   • Chat 3 is a group chat called 'Study Group'")


📋 Chats (conversations)
────────────────────────────────────────────────────────────
  id | name        | is_group | participants              
  ────────────────────────────────────────────────────────
  1  | None        | False    | alice, bob                
  2  | None        | False    | alice, charlie            
  3  | Study Group | True     | alice, bob, charlie, diana

💡 Notice:
   • Chats 1 & 2 are 1:1 chats (is_group = False, name is empty)
   • Chat 3 is a group chat called 'Study Group'


In [5]:
# 📨 Let's see the messages that already exist
messages = query("""
    SELECT m.id, m.chat_id, u.username as sender, m.content,
           m.sequence_number as seq
    FROM messages m
    JOIN users u ON m.sender_id = u.id
    ORDER BY m.chat_id, m.sequence_number
""")
print_table(messages, "All messages in the system")

print("\n💡 Key observations:")
print("   • Each message has a sequence number (seq) WITHIN its chat")
print("   • Chat 1: Alice and Bob chatting (3 messages)")
print("   • Chat 2: Alice and Charlie chatting (2 messages)")
print("   • Chat 3: Group chat with 3 messages")


📋 All messages in the system
────────────────────────────────────────────────────────────
  id | chat_id | sender  | content                              | seq
  ───────────────────────────────────────────────────────────────────
  1  | 1       | alice   | Hey Bob! How are you?                | 1  
  2  | 1       | bob     | Hi Alice! Doing great, thanks!       | 2  
  3  | 1       | alice   | Want to grab coffee later?           | 3  
  4  | 2       | alice   | Hey Charlie!                         | 1  
  5  | 2       | charlie | Hi Alice! What's up?                 | 2  
  6  | 3       | alice   | Welcome to the study group everyone! | 1  
  7  | 3       | bob     | Thanks for creating this!            | 2  
  8  | 3       | charlie | Happy to be here!                    | 3  

💡 Key observations:
   • Each message has a sequence number (seq) WITHIN its chat
   • Chat 1: Alice and Bob chatting (3 messages)
   • Chat 2: Alice and Charlie chatting (2 messages)
   • Chat 3: Group chat 

---

# 📤 Sending a Message via WebSocket

Now let's actually **send a message** through our chat system!

### What's a WebSocket?

Think of HTTP (normal web requests) like sending letters back and forth — each time you want
to say something, you write a new letter and wait for a reply.

A **WebSocket** is like a **phone call** — once connected, both sides can talk freely at any time.
This is perfect for chat apps because messages need to arrive **instantly**.

```
HTTP (like letters):              WebSocket (like a phone call):

Client ──request──► Server        Client ◄──────────► Server
Client ◄─response─ Server          (open connection, both
Client ──request──► Server           sides can send anytime)
Client ◄─response─ Server
```

### The Message Flow

Here's what happens when Alice sends a message:

```
 Alice's Phone                    Server                         Database
      │                              │                              │
      │  1. {send_message}           │                              │
      │─────────────────────────────►│                              │
      │                              │  2. Store message            │
      │                              │─────────────────────────────►│
      │                              │  3. Create inbox entries     │
      │                              │─────────────────────────────►│
      │  4. {ack, message_id}        │                              │
      │◄─────────────────────────────│                              │
      │                              │                              │
```

Let's try it! 👇

In [6]:
# 📤 Send a message as Alice to Chat 1 (Alice <-> Bob)

# Step 1: Connect as Alice (user_id = 1)
print("📱 Connecting as Alice...")
ws = ws_connect(WS_URL)
ws.send(json.dumps({"type": "connect", "user_id": 1}))
connect_resp = json.loads(ws.recv())
print(f"   ✅ Connected! Server says: {connect_resp}")

# Step 2: Send a message to chat 1
message_content = "Hey Bob! This message was sent from the Jupyter notebook! 🚀"
print(f"\n📤 Sending message: '{message_content}'")

ws.send(json.dumps({
    "type": "send_message",
    "chat_id": 1,
    "content": message_content
}))

# Step 3: Receive the ACK (acknowledgment)
ack_resp = json.loads(ws.recv())
print(f"\n📨 Server ACK received:")
print(f"   type: {ack_resp['type']}")
print(f"   message_id: {ack_resp.get('message_id')}")
print(f"   status: {ack_resp.get('status')}")

# Save the message_id so we can clean it up later
test_message_id = ack_resp.get('message_id')

ws.close()
print("\n🔌 Disconnected.")
print("\n💡 The ACK tells Alice her message was safely stored on the server.")
print("   Without this ACK, Alice's phone would keep retrying!")

📱 Connecting as Alice...
   ✅ Connected! Server says: {'type': 'connected', 'user_id': 1}

📤 Sending message: 'Hey Bob! This message was sent from the Jupyter notebook! 🚀'

📨 Server ACK received:
   type: ack
   message_id: 12
   status: stored

🔌 Disconnected.

💡 The ACK tells Alice her message was safely stored on the server.
   Without this ACK, Alice's phone would keep retrying!


In [7]:
# 🔍 Let's verify the message was stored in the database

if test_message_id:
    new_msg = query("""
        SELECT m.id, m.chat_id, u.username as sender, m.content,
               m.sequence_number as seq, m.server_timestamp
        FROM messages m
        JOIN users u ON m.sender_id = u.id
        WHERE m.id = %s
    """, (test_message_id,))
    print_table(new_msg, "Our new message in the database")

    print("\n✅ The message is safely stored!")
    print("   Even if the server crashes right now, this message is safe in PostgreSQL.")
    print(f"   Notice the sequence number — it's the next one in Chat 1.")
else:
    print("⚠️ No message was sent — check the previous cell.")


📋 Our new message in the database
────────────────────────────────────────────────────────────
  id | chat_id | sender | content                                                     | seq | server_timestamp          
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  12 | 1       | alice  | Hey Bob! This message was sent from the Jupyter notebook! 🚀 | 5   | 2026-04-20 04:19:57.796989

✅ The message is safely stored!
   Even if the server crashes right now, this message is safe in PostgreSQL.
   Notice the sequence number — it's the next one in Chat 1.


---

# 📬 The Inbox Pattern: Reliable Offline Delivery

Here's a crucial question: **what if Bob wasn't online when Alice sent her message?**

The server can't just forget about it! We need to **remember** that Bob hasn't received it yet.

This is where the **inbox table** comes in.

### How it works

Every time a message is sent, the server creates **one inbox entry per recipient**:

```
Alice sends "Hey!" to Chat 1 (Alice <-> Bob)

┌─────────────────────────────────────────────┐
│  messages table                              │
│  ┌────┬─────────┬──────┬──────┐             │
│  │ id │ chat_id │ from │ text │             │
│  │ 99 │    1    │ Alice│ Hey! │             │
│  └────┴─────────┴──────┴──────┘             │
└─────────────────────────────────────────────┘
                    │
                    ▼
┌─────────────────────────────────────────────┐
│  inbox table                                 │
│  ┌─────────┬────────────┬─────────┐         │
│  │ user_id │ message_id │ status  │         │
│  │  Bob(2) │     99     │ pending │  <-- Bob hasn't received it yet!
│  └─────────┴────────────┴─────────┘         │
└─────────────────────────────────────────────┘
```

When Bob comes online and receives the message, he sends an **ACK**.
The inbox entry is then updated to `delivered`.

### Why not just check the messages table?

Good question! The inbox table is important because:

1. **Speed**: Querying "give me all pending messages for Bob" is fast with an indexed inbox
2. **Per-user tracking**: In group chats, each person receives at their own pace
3. **Status tracking**: We know exactly what's `pending`, `delivered`, or `read`

Let's look at the inbox! 👇

In [8]:
# 📬 Let's look at the inbox — who has pending messages?

inbox = query("""
    SELECT i.id, u.username as recipient, m.content as message,
           sender.username as from_user, i.status, i.created_at
    FROM inbox i
    JOIN users u ON i.user_id = u.id
    JOIN messages m ON i.message_id = m.id
    JOIN users sender ON m.sender_id = sender.id
    WHERE i.status = 'pending'
    ORDER BY i.user_id, i.created_at
""")
print_table(inbox, "Pending inbox entries (undelivered messages)")

print("\n💡 Observations:")
print("   • Bob has pending message(s) from Alice")
print("   • Diana has 3 pending messages (she hasn't opened the group chat yet!)")
print("   • The inbox tells the server EXACTLY who needs WHICH messages")

# Also check: our new message should have created an inbox entry for Bob
if test_message_id:
    new_inbox = query("""
        SELECT u.username as recipient, i.status
        FROM inbox i
        JOIN users u ON i.user_id = u.id
        WHERE i.message_id = %s
    """, (test_message_id,))
    print(f"\n📌 Our new message (id={test_message_id}) created inbox entries for:")
    for row in new_inbox:
        print(f"   • {row['recipient']} — status: {row['status']}")


📋 Pending inbox entries (undelivered messages)
────────────────────────────────────────────────────────────
  id | recipient | message                                                     | from_user | status  | created_at                
  ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1  | bob       | Want to grab coffee later?                                  | alice     | pending | 2026-04-20 04:13:56.499690
  7  | bob       | Hey Bob! This message was sent from the Jupyter notebook! 🚀 | alice     | pending | 2026-04-20 04:19:57.810291
  2  | diana     | Welcome to the study group everyone!                        | alice     | pending | 2026-04-20 04:13:56.499690
  3  | diana     | Thanks for creating this!                                   | bob       | pending | 2026-04-20 04:13:56.499690
  4  | diana     | Happy to be here!                                           | charlie   | pending | 2026-04-


📌 Our new message (id=12) created inbox entries for:
   • bob — status: pending


---

# 📡 Redis Pub/Sub: Real-Time Delivery

The inbox handles **reliability** (making sure messages aren't lost).
But what about **speed**? When Bob IS online, we want **instant** delivery!

That's where **Redis Pub/Sub** comes in.

### What is Pub/Sub?

Imagine a **radio station**:
- The radio station **publishes** (broadcasts) a signal
- Anyone tuned in **subscribes** (listens) to that channel
- If you're not listening, you **miss** the broadcast (it's not saved)

```
         Redis Pub/Sub

  Publisher                    Subscribers
  (Server)                    (Connected users)

     📡 ─── channel: user:2 ──────► 🔊 Bob (online)     ✅ Gets it!
     │
     └── channel: user:4 ──────────► 🔇 Diana (offline)  ❌ Misses it!
                                         (but inbox has it!)
```

### Two delivery layers working together

```
┌─────────────────────────────────────────────────┐
│  Layer 1: Redis Pub/Sub (FAST but unreliable)   │
│  • Instant delivery to online users             │
│  • "At most once" — if you miss it, it's gone   │
│  • Like a live radio broadcast                  │
├─────────────────────────────────────────────────┤
│  Layer 2: Inbox (RELIABLE but slower)           │
│  • Persisted in PostgreSQL                      │
│  • Delivered on next sync                       │
│  • Like a PO Box that holds your mail           │
└─────────────────────────────────────────────────┘
```

Let's see Redis Pub/Sub in action! 👇

In [9]:
# 📡 Redis Pub/Sub in action!
#
# We'll demonstrate how the server uses Redis to notify online users.
# Channel format: "user:{user_id}"

r = get_redis()

# --- Step 1: Set up a subscriber for Bob (user_id=2) ---
# In the real system, the server does this when a user connects.
# Here we'll do it manually to see how it works.

pubsub = r.pubsub()
pubsub.subscribe("user:2")  # Listen for messages to Bob
print("📡 Subscribed to channel 'user:2' (Bob's notification channel)")

# Consume the subscription confirmation message
pubsub.get_message(timeout=1)

# --- Step 2: Publish a test notification ---
# This simulates what the server does when someone sends Bob a message
test_notification = json.dumps({
    "type": "new_message",
    "message_id": 999,
    "chat_id": 1,
    "sender_id": 1,
    "content": "This is a test notification via Redis!",
    "sequence_number": 99
})

num_receivers = r.publish("user:2", test_notification)
print(f"\n📤 Published a notification to 'user:2'")
print(f"   Number of subscribers who received it: {num_receivers}")

# --- Step 3: Read the notification as Bob ---
msg = pubsub.get_message(timeout=2)
if msg and msg['type'] == 'message':
    data = json.loads(msg['data'])
    print(f"\n🔔 Bob received a notification!")
    print(f"   Type: {data['type']}")
    print(f"   Content: {data['content']}")
    print(f"   From user_id: {data['sender_id']}")
else:
    print(f"\n⚠️ No message received (msg={msg})")

pubsub.unsubscribe("user:2")
pubsub.close()

print("\n💡 Key insight: Redis Pub/Sub is FIRE-AND-FORGET.")
print("   If Bob wasn't subscribed, the notification would be lost forever.")
print("   That's why we ALSO have the inbox table as a safety net!")

📡 Subscribed to channel 'user:2' (Bob's notification channel)

📤 Published a notification to 'user:2'
   Number of subscribers who received it: 1

🔔 Bob received a notification!
   Type: new_message
   Content: This is a test notification via Redis!
   From user_id: 1

💡 Key insight: Redis Pub/Sub is FIRE-AND-FORGET.
   If Bob wasn't subscribed, the notification would be lost forever.
   That's why we ALSO have the inbox table as a safety net!


---

# 🔄 Offline → Online Sync

Now let's see the full picture. Remember Diana? She's a member of the Study Group
(Chat 3) but hasn't been online. She has **3 pending messages** in her inbox.

Let's simulate Diana opening the app:

```
Diana's Phone                   Server                     Database
     │                              │                          │
     │  1. {connect, user_id: 4}    │                          │
     │────────────────────────────►│                          │
     │                              │                          │
     │  2. {connected}              │                          │
     │◄────────────────────────────│                          │
     │                              │                          │
     │  3. {sync}                   │  4. Query inbox          │
     │────────────────────────────►│────────────────────────►│
     │                              │                          │
     │  5. {new_message} x3         │◄────────────────────────│
     │◄────────────────────────────│                          │
     │                              │                          │
     │  6. {sync_complete, count:3} │                          │
     │◄────────────────────────────│                          │
     │                              │                          │
     │  7. {ack, message_id} x3     │  8. Update inbox         │
     │────────────────────────────►│────────────────────────►│
     │                              │     status -> delivered   │
```

Let's do this step by step! 👇

In [10]:
# 🔄 Simulate Diana coming online and syncing her messages

# First, let's see what's pending for Diana
diana_pending = query("""
    SELECT i.message_id, m.content, sender.username as from_user
    FROM inbox i
    JOIN messages m ON i.message_id = m.id
    JOIN users sender ON m.sender_id = sender.id
    WHERE i.user_id = 4 AND i.status = 'pending'
    ORDER BY m.server_timestamp
""")
print(f"📬 Diana has {len(diana_pending)} pending messages before sync:")
for msg in diana_pending:
    print(f"   • [{msg['from_user']}]: {msg['content']}")

# --- Step 1: Connect as Diana ---
print("\n📱 Diana opens the app...")
ws = ws_connect(WS_URL)
ws.send(json.dumps({"type": "connect", "user_id": 4}))
connect_resp = json.loads(ws.recv())
print(f"   ✅ Connected! ({connect_resp})")

# --- Step 2: Request sync ---
print("\n🔄 Diana requests sync (give me everything I missed)...")
ws.send(json.dumps({"type": "sync"}))

# --- Step 3: Receive all pending messages ---
synced_messages = []
while True:
    resp = json.loads(ws.recv())
    if resp["type"] == "sync_complete":
        print(f"\n✅ Sync complete! Received {resp['count']} messages.")
        break
    elif resp["type"] == "new_message":
        synced_messages.append(resp)
        print(f"   📨 Message {resp['message_id']}: \"{resp['content']}\"")

# --- Step 4: ACK each message ---
print("\n📝 Diana ACKs each message (confirming delivery)...")
for msg in synced_messages:
    ws.send(json.dumps({"type": "ack", "message_id": msg["message_id"]}))
    ack_resp = json.loads(ws.recv())
    print(f"   ✅ ACK'd message {msg['message_id']} -> {ack_resp['type']}")

ws.close()
print("\n🔌 Diana disconnects.")

📬 Diana has 3 pending messages before sync:
   • [alice]: Welcome to the study group everyone!
   • [bob]: Thanks for creating this!
   • [charlie]: Happy to be here!

📱 Diana opens the app...
   ✅ Connected! ({'type': 'connected', 'user_id': 4})

🔄 Diana requests sync (give me everything I missed)...
   📨 Message 6: "Welcome to the study group everyone!"
   📨 Message 7: "Thanks for creating this!"
   📨 Message 8: "Happy to be here!"

✅ Sync complete! Received 3 messages.

📝 Diana ACKs each message (confirming delivery)...
   ✅ ACK'd message 6 -> ack_ok
   ✅ ACK'd message 7 -> ack_ok


   ✅ ACK'd message 8 -> ack_ok

🔌 Diana disconnects.


In [11]:
# 🔍 Let's verify Diana's inbox was updated

diana_inbox = query("""
    SELECT i.message_id, m.content, i.status, i.delivered_at
    FROM inbox i
    JOIN messages m ON i.message_id = m.id
    WHERE i.user_id = 4
    ORDER BY i.message_id
""")
print_table(diana_inbox, "Diana's inbox after sync")

print("\n💡 All messages are now 'delivered'!")
print("   The inbox entries were updated when Diana sent her ACKs.")
print("   If Diana syncs again, she won't get these messages — they're already delivered.")


📋 Diana's inbox after sync
────────────────────────────────────────────────────────────
  message_id | content                              | status    | delivered_at              
  ──────────────────────────────────────────────────────────────────────────────────────────
  6          | Welcome to the study group everyone! | delivered | 2026-04-20 04:19:57.966361
  7          | Thanks for creating this!            | delivered | 2026-04-20 04:19:57.972461
  8          | Happy to be here!                    | delivered | 2026-04-20 04:19:57.977418

💡 All messages are now 'delivered'!
   The inbox entries were updated when Diana sent her ACKs.
   If Diana syncs again, she won't get these messages — they're already delivered.


---

# 🔢 Sequence Numbers: Detecting Missed Messages

Imagine you're watching a TV series, and you see episodes 1, 2, 3, 5.
You'd immediately know: **"Wait, I missed episode 4!"**

That's exactly how **sequence numbers** work in our chat system.

### How it works

Each chat has a **counter** that increases by 1 for every message:

```
Chat 1 (Alice <-> Bob):

  seq=1: "Hey Bob!"           ✅ Client has this
  seq=2: "Hi Alice!"          ✅ Client has this
  seq=3: "Want coffee?"       ❌ Client missed this!
  seq=4: "Sure, when?"        ✅ Client has this
                                     │
                                     ▼
                              🚨 GAP DETECTED!
                              Client knows seq 3 is
                              missing and requests it.
```

### Why is this needed?

Even with the inbox, things can go wrong:
- Network hiccup during delivery
- Server crashes between storing message and notifying
- Race conditions in distributed systems

Sequence numbers give the **client** a way to detect and fix problems **independently**.

Let's look at the actual sequences! 👇

In [12]:
# 🔢 Let's examine chat sequence numbers

sequences = query("""
    SELECT cs.chat_id,
           COALESCE(c.name, 'Chat ' || cs.chat_id::text) as chat_name,
           cs.last_sequence
    FROM chat_sequences cs
    JOIN chats c ON cs.chat_id = c.id
    ORDER BY cs.chat_id
""")
print_table(sequences, "Chat sequence counters")

print("\n💡 Each chat tracks how many messages have been sent.")
print("   These counters only go UP — they never reset or go backwards.")


📋 Chat sequence counters
────────────────────────────────────────────────────────────
  chat_id | chat_name   | last_sequence
  ─────────────────────────────────────
  1       | Chat 1      | 5            
  2       | Chat 2      | 2            
  3       | Study Group | 3            

💡 Each chat tracks how many messages have been sent.
   These counters only go UP — they never reset or go backwards.


In [13]:
# 🔍 Simulating gap detection on the client side
#
# A real client would track the last sequence number it received per chat.
# Let's simulate what a client does to detect gaps.

def detect_gaps(chat_id):
    """Check if any sequence numbers are missing in a chat."""
    messages = query("""
        SELECT sequence_number FROM messages
        WHERE chat_id = %s
        ORDER BY sequence_number
    """, (chat_id,))

    if not messages:
        return []

    seq_numbers = [m['sequence_number'] for m in messages]
    expected = set(range(1, max(seq_numbers) + 1))
    actual = set(seq_numbers)
    gaps = sorted(expected - actual)
    return gaps

# Check all chats
for chat_id in [1, 2, 3]:
    gaps = detect_gaps(chat_id)
    msgs = query("""
        SELECT sequence_number as seq, content
        FROM messages WHERE chat_id = %s
        ORDER BY sequence_number
    """, (chat_id,))

    chat_info = query(
        "SELECT COALESCE(name, 'Chat ' || id::text) as name FROM chats WHERE id = %s",
        (chat_id,)
    )
    chat_name = chat_info[0]['name']

    print(f"\n{'─' * 50}")
    print(f"📋 {chat_name} (chat_id={chat_id})")
    print(f"   Sequence numbers: {[m['seq'] for m in msgs]}")
    if gaps:
        print(f"   🚨 GAPS DETECTED: Missing sequences {gaps}")
        print(f"   -> Client would request these from the server")
    else:
        print(f"   ✅ No gaps — all messages accounted for!")

print("\n💡 In a real app, the client checks for gaps on every sync.")
print("   If gaps are found, it requests the missing messages from the server.")
print("   This is the LAST line of defense for message reliability!")


──────────────────────────────────────────────────
📋 Chat 1 (chat_id=1)
   Sequence numbers: [1, 2, 3, 5]
   🚨 GAPS DETECTED: Missing sequences [4]
   -> Client would request these from the server



──────────────────────────────────────────────────
📋 Chat 2 (chat_id=2)
   Sequence numbers: [1, 2]
   ✅ No gaps — all messages accounted for!



──────────────────────────────────────────────────
📋 Study Group (chat_id=3)
   Sequence numbers: [1, 2, 3]
   ✅ No gaps — all messages accounted for!

💡 In a real app, the client checks for gaps on every sync.
   If gaps are found, it requests the missing messages from the server.
   This is the LAST line of defense for message reliability!


---

# 🧹 Cleanup

Let's clean up the test data we created during this notebook so the database
is back to its original state for the next notebook.

In [14]:
# 🧹 Clean up test data created during this notebook

if test_message_id:
    # Delete inbox entries for our test message
    execute("DELETE FROM inbox WHERE message_id = %s", (test_message_id,))
    print(f"🗑️  Deleted inbox entries for message {test_message_id}")

    # Delete the test message itself
    execute("DELETE FROM messages WHERE id = %s", (test_message_id,))
    print(f"🗑️  Deleted test message {test_message_id}")

    # Reset chat 1's sequence counter back to 3
    execute("UPDATE chat_sequences SET last_sequence = 3 WHERE chat_id = 1")
    print(f"🔄 Reset Chat 1 sequence counter to 3")
else:
    print("ℹ️  No test data to clean up.")

# Reset Diana's inbox back to pending (we ACK'd them during sync demo)
execute("""
    UPDATE inbox
    SET status = 'pending', delivered_at = NULL
    WHERE user_id = 4 AND message_id IN (6, 7, 8)
""")
print("🔄 Reset Diana's inbox entries back to 'pending'")

print("\n✅ Cleanup complete! Database is back to its original state.")

🗑️  Deleted inbox entries for message 12


🗑️  Deleted test message 12
🔄 Reset Chat 1 sequence counter to 3
🔄 Reset Diana's inbox entries back to 'pending'

✅ Cleanup complete! Database is back to its original state.


---

# 📝 Summary: Key Takeaways

Congratulations! You've explored the core of how a messaging system delivers messages. 🎉

### What we learned

| Concept | What it does | Real-world analogy |
|---------|-------------|-------------------|
| **WebSocket** | Real-time bidirectional connection | Phone call |
| **Messages table** | Permanent storage of all messages | Filing cabinet |
| **Inbox table** | Tracks what each user needs to receive | PO Box |
| **ACK (Acknowledgment)** | Confirms message was delivered | Signed receipt |
| **Redis Pub/Sub** | Instant notification to online users | Radio broadcast |
| **Sync** | Catch up on missed messages | Checking your PO Box |
| **Sequence Numbers** | Detect missed messages | Episode numbers on TV |

### The two-layer delivery system

```
                    Message Sent
                        │
                ┌───────┴───────┐
                │               │
                ▼               ▼
        ┌──────────────┐ ┌──────────────┐
        │  Redis       │ │   Inbox      │
        │  Pub/Sub     │ │  (Database)  │
        │              │ │              │
        │  Fast but    │ │  Slow but    │
        │  unreliable  │ │  reliable    │
        │  (at most    │ │  (at least   │
        │   once)      │ │   once)      │
        └──────────────┘ └──────────────┘
                │               │
                └───────┬───────┘
                        │
                        ▼
               Together = Reliable
               AND fast delivery! ✅
```

### 🔮 What's Next?

In **Notebook 2: Read Receipts & Presence**, we'll explore:
- How the ✓✓ (double checkmark) system works
- How the app knows when someone is "online" or "last seen at..."
- How read receipts flow back to the sender

See you there! 👋